## Key Parameters

Parameters like temperature and top_p control randomness and creativity of output.
Max tokens limits response length and helps control cost.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI


load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
    model="gpt-4o",
    input="Write a creative AI tagline",
    temperature=1.0,
    max_output_tokens=50,
    top_p=1.0
    # it is a parameter that controls the diversity of the generated output. A higher value (e.g., 0.9) will make the output more diverse, while a lower value (e.g., 0.1) will make it more focused and deterministic.
)

print(response.output_text)

"Empower Imagination, Elevate Innovation."


## Inference Techniques


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
    model="gpt-4o",
    input="Tell a joke about coding",
    temperature=1.0
)

print(response.output_text)

Why do programmers prefer dark mode?

Because light attracts bugs!


## Tokens & Context Window

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
    model="gpt-4o",
    input=[
         {
            "role": "system",
            "content": "You are a helpful assistant."
        },
         {
        "role": "user", 
        "content": "What is the capital of France?"
    }]
)

print(response.output_text)

The capital of France is Paris.


## What is Tool Calling?

Tool calling allows the LLM to decide when to call external functions (tools) instead of answering directly with the pretrained memeroy that LLM already has.

In [4]:
def get_weather(city: str):
    # This is a placeholder function. In a real implementation, you would call a weather API here.
    return f"The current weather in {city} is sunny with a temperature of 25°C."

In [5]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# Load env
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ✅ Step 1: Define actual function
def get_weather(city: str):
    return f"The weather in {city} is 35°C and sunny."

# ✅ Step 2: Define tool schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# ✅ Step 3: First call (LLM decides tool)
response = client.responses.create(
    model="gpt-4o-mini",
    tools=tools,
    input="What is the weather in Kanpur?"
)

# ✅ Step 4: Extract tool call safely
tool_calls = response.output[0].content

for item in tool_calls:
    if item.type == "tool_call":
        tool_name = item.name
        arguments = json.loads(item.arguments)

        # ✅ Step 5: Execute function
        result = get_weather(arguments["city"])

        # ✅ Step 6: Send tool output back
        final_response = client.responses.create(
            model="gpt-4o-mini",
            tools=tools,
            input=[
                {
                    "role": "user",
                    "content": "What is the weather in Kanpur?"
                },
                item,
                {
                    "type": "tool_output",
                    "tool_name": tool_name,
                    "content": result
                }
            ]
        )

        print(final_response.output[0].content[0].text)

BadRequestError: Error code: 400 - {'error': {'message': "Missing required parameter: 'tools[0].name'.", 'type': 'invalid_request_error', 'param': 'tools[0].name', 'code': 'missing_required_parameter'}}